# Vessel detection in the port of Constanța with Sentinel-1

We count ships in Constanta port and its offshore anchorage using Sentinel-1 radar images (2019-2025) in Google Earth Engine

## **Research questions**
1. How many vessels can be detected with backscatter thresholding?
2. How did vessel counts change after the Russia-Ukraine war started (24 Feb 2022)
3. How well do our counts agree with IMF PortWatch port calls?

## Settings
All options for the analysis are in one place, so you can change them without searching through the notebook.

In [51]:
# Google Earth Engine project (change to your own)
gee_project = 'tutorial-geoscripting-paulian'

# Constanța port plus the offshore anchorage zone analysis area 
bbox = [28.64, 44.08, 28.88, 44.18]
split_lon = 28.70
# Time Interval
start_date = '2019-01-01'
end_date   = '2026-01-01'   

# Specifying the most frequent orbit
orbit = 58

# Ship detection
threshold = -15
k = 4
guard = 20
outer = 25
min_pixels = 8

# File paths
water_shp = 'masks/clip_water.shp'
data_dir = 'data'
output_dir = 'output'

## 2. Setup
Load the packages, connect to Google Earth Enginge and create the folders for results.

In [54]:
import os
import ee
import geemap
from functions.detection import load_water, load_s1, detect_ships, ship_counts, sea_threshold
import pandas as pd
import matplotlib.pyplot as plt

# Log in to Google Earth Engine (only need to do it once per environment)
# ee.Authenticate()

# Connect to Google Earth Engine
ee.Initialize(project=gee_project)

# Create the data and output folders if they don't exist yet
os.makedirs(data_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# 3. Water mask
In radar images, ships light up amongst bodies of water, but so does land, buildings and docks. To only count ships we will look at water pixels only.

We use the OpenStreetMap water polygons ([osmdata.openstreetmap.de](https://osmdata.openstreetmap.de/data/water-polygons.html)), clipped to our area of interest.

The mask shrinks a few metres inward, so the bright edges of land and docks are not mistaken for ships.

In [55]:
# https://geemap.org/cheatsheet/#add-earth-engine-layers
# https://geoscripting-wur.github.io/Python-GEE/


water = load_water(water_shp, bbox, buffer=10)
port = ee.Geometry.Rectangle([bbox[0], bbox[1], split_lon, bbox[3]])
anchorage = ee.Geometry.Rectangle([split_lon, bbox[1], bbox[2], bbox[3]])
m = geemap.Map()
m.centerObject(water, 12)
m.addLayer(water, {'color': 'blue'}, ' water mask')
m

Map(center=[44.12994105458297, 28.76675569965496], controls=(WidgetControl(options=['position', 'transparent_b…

## 4. Sentinel-1 images
We use Sentinel-1 radar images with the VH band. VH gives more contrast between ships and water than VV.


In [56]:
aoi = ee.Geometry.Rectangle(bbox)
s1 = load_s1(aoi, start_date, end_date, orbit)
print('Number of Sentinel-1 scenes:', s1.size().getInfo())

Number of Sentinel-1 scenes: 322


## 5. Ship detection on one scene
To recognize ships in an image, we are testing two methods to find them.
1. **Fixed threshold:** every water pixel brighter than 'threshold' is a ship pixel
2. **CFAR (Constant False Alarm Rate):** a pixel is a ship if it is much brighter than the water around it

Touching bright pixels will form one object, and objects smaller than min_pixels are seen as noise and will be removed.

In [57]:
image = s1.first()
print('Date', image.date().format('YYYY-MM-dd HH:mm').getInfo())
ships_threshold = detect_ships(image, water, method='threshold', threshold=threshold, min_pixels=min_pixels)
ships_cfar = detect_ships(image, water, method='cfar', k=2, min_pixels=min_pixels)

print('Ships threshold:', ships_threshold.size().getInfo())
print('Ships CFAR:', ships_cfar.size().getInfo())

Date 2019-01-04 16:00
Ships threshold: 152
Ships CFAR: 129


In [60]:
ships = detect_ships(image, water, method='threshold', threshold=threshold, min_pixels=min_pixels)
ships_port = ships.filterBounds(port)
ships_anchorage = ships.filterBounds(anchorage)
print('Port:', ships_port.size().getInfo(), '| Anchorage:', ships_anchorage.size().getInfo())

map = geemap.Map()
map.centerObject(water, 12)
map.addLayer(water, {'color': 'blue'}, 'water mask')
map.addLayer(ships_port, {'color': 'red'}, 'ships - port')
map.addLayer(ships_anchorage, {'color': 'yellow'}, 'ships - anchorage')
map

Port: 147 | Anchorage: 5


Map(center=[44.12994105458665, 28.76675569965353], controls=(WidgetControl(options=['position', 'transparent_b…

## 6. Time series
We count the ships in every scence from 2019 to 2025.
This process will take a few minutes, so the result is saved to 'data/ship_counts.csv. 
THe next time you run the notebook, the counts are loaded from that file instead.

In [59]:
# Ocounts_cfar = ship_counts(s1, water, method='cfar', guard=guard, outer=outer, k=k, min_pixels=min_pixels)
counts = ship_counts(s1, water, threshold=threshold, k=k, min_pixels=min_pixels)


EEException: Too many concurrent aggregations.

## 7. Before and after the war
Monthly average number of detected ships for both methods. 

In [ ]:
# Monthly average, so the plot is less noisy than 322 single dates
monthly = counts.resample('MS').mean()
war_start = '2022-02-24'

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly.index, monthly['n_fixed'], label='Fixed threshold')
ax.plot(monthly.index, monthly['n_adaptive'], label='Fixed threshold')

ax.axvline(pd.Timestamp(war_start), color='black', linestyle='--', label='Start of war')
ax.set_ylabel('Detected ships (monthly mean)')
ax.legend()
fig.savefig(os.path.join(output_dir, 'ship_counts.png'), dpi=150, bbox_inches='tight')
plt.show()

# Average number of ships per image, before and after the start of the war
before = counts[counts.index < war_start].mean()
after = counts[counts.index >= war_start].mean()
print(pd.DataFrame({'before war': before, 'after war': after}).round(1))

counts.to_csv(os.path.join(data_dir, 'ship_counts.csv'))